In [30]:
from neo4j import GraphDatabase
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline
from neo4j_graphrag.llm import OpenAILLM, OllamaLLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.generation.prompts import ERExtractionTemplate
from dotenv import load_dotenv
import pandas as pd
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from neo4j_graphrag.retrievers import VectorRetriever, VectorCypherRetriever, Text2CypherRetriever
from neo4j_graphrag.schema import get_schema
from neo4j_graphrag.generation import GraphRAG


import os, time, asyncio, glob, csv

## 1. Initialize Graph Database 

In [78]:
load_dotenv(dotenv_path="../.env")

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = os.getenv("NEO4J_DATABASE")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

### 1.1 Call BGE-M3 Embedding model

In [3]:
import torch
from FlagEmbedding import BGEM3FlagModel

# เช็คว่ามี GPU (CUDA) หรือไม่
# ถ้ามีให้ใช้ 'cuda' ถ้าไม่มีให้ใช้ 'cpu'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_fp16 = True if device == 'cuda' else False  # fp16 ส่วนใหญ่รองรับเฉพาะบน GPU

print(f"กำลังรันโมเดลบน: {device.upper()}")

# 2. Initialize Model
embedder = BGEM3FlagModel(
    'VISAI-AI/nitibench-ccl-human-finetuned-bge-m3', 
    use_fp16=use_fp16, 
    device=device 
)

กำลังรันโมเดลบน: CUDA


Fetching 23 files: 100%|██████████| 23/23 [00:00<?, ?it/s]


### 1.2 Wrap embedding model
เนื่องจาก Embedding model ที่จะใช้ใน Neo4J (เรียกจาก library Neo4J โดยตรง ไม่ใช่ langchain) ไม่สามารถเรียกใช้ Embedding model BGE-M3 ได้โดยตรง ตัว Library รองรับแค่พวก Commercial Models e.g. openai จึงต้องมีการ Wrap เพื่อให้ใช้งานได้ 

In [4]:
from neo4j_graphrag.embeddings import Embedder

class Neo4jCompatibleEmbedder(Embedder):
    def __init__(self, original_embedder):
        self.original_embedder = original_embedder

    def embed_query(self, text: str) -> list[float]:
        # 1. เรียกโมเดล M3 (สมมติว่าใช้ .encode)
        result = self.original_embedder.encode(text)
        
        # 2. ตรวจสอบว่าผลลัพธ์เป็น dict หรือไม่ (BGE-M3 มักคืนค่าแบบนี้ถ้าไม่ได้ตั้ง return_dense_only=True)
        if isinstance(result, dict):
            # ดึงเฉพาะ Dense Vector ออกมา
            embedding = result.get('dense_vecs')
        else:
            embedding = result

        # 3. แปลงจาก numpy array เป็น list เพื่อให้ Pydantic/Neo4j ยอมรับ
        if hasattr(embedding, "tolist"):
            return embedding.tolist()
        
        # กรณีเป็น list อยู่แล้วแต่สมาชิกข้างในอาจเป็น numpy types ให้วนลูปแปลง (ถ้าจำเป็น)
        return [float(x) for x in embedding]

    def embed_nodes(self, nodes):
        for node in nodes:
            # node.text คือเนื้อหาที่ถูก Chunk แล้ว
            node.embedding = self.embed_query(node.text)

# การใช้งาน
# ระวังอย่ารันบรรทัดนี้ซ้ำ (เพราะจะกลายเป็น wrap ซ้อน wrap)
neo4j_embedder = Neo4jCompatibleEmbedder(embedder)

In [ ]:
df = pd.read_parquet('../data/processed/nitibench_cleaned_2026-03-17.parquet')
df.head()

## 2. Chunking to documents

In [48]:
START = 100
END = 150

def batch_document_append(df,start=0, end=len(df)):
    documents = []
    for index, row in df.iloc[start:end,:].iterrows():
        # สร้างเนื้อหาที่มีบริบทชัดเจนในตัวเอง
        content = f"กฎหมาย: {row['law_name']}\nเนื้อหา: {row['section_content']}"
        
        # เก็บ Metadata เพื่อใช้ในภายหลัง
        metadata = {
            "law_name": row['law_name'],
        }
        documents.append(Document(page_content=content, metadata=metadata))
    return documents

documents = batch_document_append(df,start=START, end=END)
documents

[Document(metadata={'law_name': 'พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535'}, page_content='กฎหมาย: พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535\nเนื้อหา: พระราชบัญญัติหลักทรัพย์และตลาดหลักทรัพย์ พ.ศ. 2535 มาตรา 189 บุคคลใดประสงค์จะนำหลักทรัพย์ที่ตนออกไปซื้อขายในตลาดหลักทรัพย์ จะต้องนำหลักทรัพย์นั้นไปจดทะเบียนกับตลาดหลักทรัพย์\nเมื่อตลาดหลักทรัพย์ได้รับคำขอจดทะเบียนแล้ว ให้พิจารณาและเสนอความเห็นต่อคณะกรรมการตลาดหลักทรัพย์เพื่อสั่งรับหรือไม่รับเป็นหลักทรัพย์จดทะเบียน'),
 Document(metadata={'law_name': 'พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535'}, page_content='กฎหมาย: พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535\nเนื้อหา: พระราชบัญญัติบริษัทมหาชนจำกัด พ.ศ. 2535 มาตรา 205 บริษัทใดไม่ปฏิบัติตามมาตรา 109 มีความผิดทางพินัยต้องชำระค่าปรับเป็นพินัยไม่เกินสองแสนบาท และชำระค่าปรับเป็นพินัยรายวันอีกวันละสองพันบาท*จนกว่าจะปฏิบัติถูกต้อง'),
 Document(metadata={'law_name': 'พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ.ศ. 2550'}, page_content='กฎหมาย: พระราชบัญญัติทรัสต์เพื่อธุรกรรมในตลาดทุน พ

## 3. Select LLM

In [23]:
from neo4j_graphrag.llm import OllamaLLM
# llm = OllamaLLM(
#     model_name="llama3",
#     # model_params={"options": {"temperature": 0}, "format": "json"},
#     host="http://localhost:11434",  # when using a remote server
# )

os.environ["OPENAI_API_KEY"] = os.getenv("openrouter_API_key") # ใส่ OpenRouter Key ของคุณตรงนี้

# 2. เรียกใช้งาน โดยใส่ base_url กำกับไว้ข้างนอก
llm = OpenAILLM(
    model_name="openai/gpt-4o-mini", 
    base_url="https://openrouter.ai/api/v1",
    model_params={
        "temperature": 0
    }
)
# llm.invoke("say hello-world")

## 4. Graph Construction

In [49]:
# llm = ChatOpenAI(api_key=os.getenv("openrouter_API_key") ,temperature=0, model="openai/gpt-4o-mini", base_url="https://openrouter.ai/api/v1")
# llm = OllamaLLM(model="llama3", )

entities = [
    {"label": "law_name", "properties": [{"name": "text", "type": "STRING"}]},
    {"label": "section_number", "properties": [{"name": "text", "type": "STRING"}]},
    {"label": "concept", "properties": [{"name": "text", "type": "STRING"}]},
]

relations = [
    {"label": "CONTAINS", "source": "law_name", "target": "section_number"},
    {"label": "REFERENCES", "source": "section_number", "target": "section_number"},
    {"label": "MENTIONS", "source": "section_number", "target": "concept"},
]

kg_builder = SimpleKGPipeline(
    driver=driver,
    llm=llm,
    embedder=neo4j_embedder,
    entities=entities,
    relations=relations,
    neo4j_database=DATABASE_NAME,
    from_pdf=False,
    )

In [50]:
import time
from langchain_community.callbacks import get_openai_callback

start_time = time.perf_counter()
with get_openai_callback() as cb:
    for i in range(len(documents)):
        #นำเข้า Neo4j ทีละ Document (ซึ่งแต่ละ Document ก็คือ Section ที่ถูก Chunk แล้ว) สิ่งที่นำเข้าเป็น json format
        await kg_builder.run_async(text=documents[i].page_content, document_metadata=documents[i].metadata,) 
end_time = time.perf_counter()
elapsed_time = end_time - start_time
print(f"Elapsed time: {elapsed_time:.4f} seconds")
print(f"Total Tokens: {cb.total_tokens}")
print(f"Prompt Tokens: {cb.prompt_tokens}")
print(f"Completion Tokens: {cb.completion_tokens}")

LLM response has improper format for chunk_index=0


Elapsed time: 497.3945 seconds
Total Tokens: 0
Prompt Tokens: 0
Completion Tokens: 0


## 5. Retrieval

### 5.1 Vector Retriever เหมือน RAG ธรรมดา

In [ ]:
# Initialize the retriever
retriever = VectorRetriever(
     driver,
     index_name= "text_embeddings",
     embedder=neo4j_embedder,
     return_properties=["text"]
)
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร"
result = retriever.search(query_text=query, top_k=5)

In [25]:
# ดึงเนื้อหาของอันแรก
first_result = result.items[0].content
# ดึงค่า Score (ความเหมือน)
first_score = result.items[0].metadata['score']

print(f"Score: {first_score}")
print(f"Content: {first_result}")

Score: 0.9035258293151855
Content: {'text': 'กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nมาตรา: 132\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน'}


### 5.2 Vector Cypher Retriever 
- ใช้ Cosine similarity เทียบความคล้าย
- Cypher query เข้าไป (ต้องมีการกำหนดเอง ดีที่สุดคืออิงจาก Schema) 

In [95]:
schema = get_schema(driver)
schema

'Node properties:\nDocument {document_type: STRING, path: STRING, createdAt: STRING, law_name: STRING}\nChunk {index: INTEGER, text: STRING, embedding: LIST}\nlaw_name {text: STRING}\nsection_number {text: STRING}\nconcept {text: STRING}\nRelationship properties:\n\nThe relationships:\n(:Chunk)-[:FROM_DOCUMENT]->(:Document)\n(:law_name)-[:FROM_CHUNK]->(:Chunk)\n(:law_name)-[:CONTAINS]->(:section_number)\n(:law_name)-[:REFERENCES]->(:section_number)\n(:law_name)-[:REFERENCES]->(:law_name)\n(:law_name)-[:MENTIONS]->(:section_number)\n(:law_name)-[:MENTIONS]->(:concept)\n(:section_number)-[:FROM_CHUNK]->(:Chunk)\n(:section_number)-[:CONTAINS]->(:concept)\n(:section_number)-[:MENTIONS]->(:concept)\n(:section_number)-[:MENTIONS]->(:section_number)\n(:section_number)-[:REFERENCES]->(:concept)\n(:section_number)-[:REFERENCES]->(:section_number)\n(:section_number)-[:REFERENCES]->(:law_name)\n(:concept)-[:FROM_CHUNK]->(:Chunk)\n(:concept)-[:MENTIONS]->(:concept)\n(:concept)-[:REFERENCES]->(:con

In [ ]:
chunk_to_law_query = """
MATCH (node)-[:FROM_DOCUMENT]->(d:Document)
        OPTIONAL MATCH (s:section_number)-[:FROM_CHUNK]->(node)
        RETURN 
            "Law: " + d.law_name + " Section: " + s.text + " Content: " + node.text AS content, 
            score, 
            {id: node.index} AS metadata
"""

vector_cypher_retriever = VectorCypherRetriever(
    driver=driver,
    index_name="text_embeddings",
    embedder=neo4j_embedder,
    retrieval_query=chunk_to_law_query
)

query = "ขับรถชนกันโดยประมาทร่วมกัน ต้องใช้ค่าเสียหายเท่ากันทั้งสองฝ่ายใช่หรือไม่" # ข้อ 38
result = vector_cypher_retriever.search(query_text=query, top_k=5)
for item in result.items:
    print(item.content)

<Record content='Law: ประมวลกฎหมายแพ่งและพาณิชย์ Section: 223 Content: กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 223\nถ้าฝ่ายผู้เสียหายได้มีส่วนทำความผิดอย่างใดอย่างหนึ่งก่อให้เกิดความเสียหายด้วยไซร้ ท่านว่าหนี้อันจะต้องใช้ค่าสินไหมทดแทนแก่ฝ่ายผู้เสียหายมากน้อยเพียงใดนั้นต้องอาศัยพฤติการณ์เป็นประมาณ ข้อสำคัญก็คือว่าความเสียหายนั้นได้เกิดขึ้นเพราะฝ่ายไหนเป็นผู้ก่อยิ่งหย่อนกว่ากันเพียงไร\nวิธีเดียวกันนี้ ท่านให้ใช้แม้ทั้งที่ความผิดของฝ่ายผู้ที่เสียหายจะมีแต่เพียงละเลยไม่เตือนลูกหนี้ให้รู้สึกถึงอันตรายแห่งการเสียหายอันเป็นอย่างร้ายแรงผิดปกติ ซึ่งลูกหนี้ไม่รู้หรือไม่อาจจะรู้ได้ หรือเพียงแต่ละเลยไม่บำบัดปัดป้อง หรือบรรเทาความเสียหายนั้นด้วย อนึ่งบทบัญญัติแห่งมาตรา 220 นั้นท่านให้นำมาใช้บังคับด้วยโดยอนุโลม' score=0.7881317138671875 metadata={'id': 0}>
<Record content='Law: ประมวลกฎหมายแพ่งและพาณิชย์ Section: 442 Content: กฎหมาย: ประมวลกฎหมายแพ่งและพาณิชย์\nเนื้อหา: ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 442\nถ้าความเสียหายได้เกิดขึ้นเพราะความผิดอย่างหนึ่งอย่างใด

In [102]:
query = "ถ้ามีคนประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตต้องระวางโทษอย่างไร" # ข้อ 1
result = vector_cypher_retriever.search(query_text=query, top_k=5)
for item in result.items:
    print(item.content)

<Record content='Law: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 Section: 132 Content: กฎหมาย: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546\nเนื้อหา: พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน' score=0.9024844169616699 metadata={'id': 0}>
<Record content='Law: พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542 Section: 37 Content: กฎหมาย: พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542\nเนื้อหา: พระราชบัญญัติการประกอบธุรกิจของคนต่างด้าว พ.ศ. 2542 มาตรา 37 คนต่างด้าวผู้ใดประกอบธุรกิจโดยฝ่าฝืนมาตรา 6 มาตรา 7 หรือมาตรา 8 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับตั้งแต่หนึ่งแสนบาทถึงหนึ่งล้านบาท หรือทั้งจำทั้งปรับ และให้ศาลสั่งเลิกการประกอบธุรกิจ หรือเลิกกิจการ หรือสั่งเลิกการเป็นผู้ถือหุ้น หรือเป็นหุ้นส่วน แล้วแต่กรณี หากฝ่าฝืนไม่ปฏิบัติต

## 6. ใช้ Typhoon QA

In [83]:
os.environ["OPENAI_API_KEY"] = os.getenv("thai_llm_API_key")

model = "typhoon"
# 2. สร้าง LLM Object ให้ถูกต้องสำหรับ neo4j-graphrag
llm = OpenAILLM(
    model_name="typhoon-v2.5-30b-a3b-instruct",
    # สำคัญ: base_url ต้องหยุดอยู่ที่ /v1 (ห้ามมี /chat/completions ต่อท้าย)
    base_url=f"https://api.opentyphoon.ai/v1",
    # ต้องระบุ api_key เป็น string เสมอ (แม้จะเป็นค่าว่างก็ต้องใส่ "") เพื่อป้องกัน TypeError
    model_params={
        "temperature": 0.1,
        "max_tokens": 4096,
    }
)
# llm.invoke("สวัสดี")

### คำตอบข้อ 1

In [103]:
result = GraphRAG(llm=llm,retriever=vector_cypher_retriever)
print(result.search(query_text=query, retriever_config= {"top_k": 5}).answer)

ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน.


### คำตอบข้อ 38 

In [89]:
result = GraphRAG(llm=llm,retriever=vector_cypher_retriever)
print(result.search(query_text=query, retriever_config= {"top_k": 5}).answer)

ไม่จำเป็นต้องใช้ค่าเสียหายเท่ากันทั้งสองฝ่ายเสมอไป

ตามประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 223 ระบุว่า ถ้าฝ่ายผู้เสียหายมีส่วนทำความผิดก่อให้เกิดความเสียหาย ท่านว่าหนี้อันจะต้องใช้ค่าสินไหมทดแทนแก่ฝ่ายผู้เสียหายมากน้อยเพียงใดนั้น ต้องอาศัยพฤติการณ์เป็นประมาณ โดยพิจารณาจากว่า ความเสียหายนั้นเกิดขึ้นเพราะฝ่ายใดเป็นผู้ก่อให้เกิดความผิดได้ยิ่งหย่อนกว่ากันเพียงใด

ในกรณีที่ขับรถชนกันโดยประมาทร่วมกัน หมายความว่าทั้งสองฝ่ายมีความผิดร่วมกัน แต่ระดับความผิดและความรุนแรงของพฤติกรรมอาจแตกต่างกัน เช่น ฝ่ายหนึ่งขับเร็วเกินกว่ากำหนด ขณะที่อีกฝ่ายอาจเลี้ยวข้ามเลนโดยไม่สังเกต ดังนั้น ศาลจะพิจารณาความผิดของแต่ละฝ่ายอย่างละเอียด และกำหนดให้แต่ละฝ่ายรับผิดชอบค่าเสียหายในสัดส่วนที่เหมาะสมกับความผิดของตนเอง ไม่จำเป็นต้องแบ่งเท่ากันเสมอไป

นอกจากนี้ มาตรา 442 ยังระบุว่า ถ้าความเสียหายเกิดจากความผิดของผู้ต้องเสียหายประกอบด้วย ให้นำมาตรา 223 มาใช้บังคับโดยอนุโลม แสดงว่าหลักการลดค่าสินไหมทดแทนตามสัดส่วนความผิดของผู้เสียหาย ยังสามารถนำมาใช้ได้ในกรณีความผิดร่วมกัน

ดังนั้น การขับรถชนกันโดยประมาทร่วมกัน ไม่ได้หมายคว